<a href="https://colab.research.google.com/github/dbh25-10120-cloud/scaling-waffle/blob/main/%EB%85%BC%EB%AC%B8_%EC%9A%94%EC%95%BD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pymupdf transformers torch scikit-learn sentencepiece

In [2]:
import fitz

pdf_path = "/content/000002067894_20260330212044.pdf"

def extract_text(pdf_path):
    doc = fitz.open(pdf_path)
    return "".join([page.get_text() for page in doc])

def extract_text(pdf_path):
    doc = fitz.open(pdf_path)
    return "".join([page.get_text() for page in doc])

paper_text = extract_text(pdf_path)

In [3]:
import re

def split_sentences(text):
    return re.split(r'(?<=[.!?다])\s+', text)

sentences = split_sentences(paper_text)

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# GPU 사용 (가능하면)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


def embed(sentences, batch_size=8, max_length=128):
    all_embeddings = []

    for i in range(0, len(sentences), batch_size):
        batch = sentences[i:i+batch_size]

        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,   # 🔥 토큰 길이 제한
            return_tensors="pt"
        )

        # GPU로 이동
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)

        batch_embeddings = outputs.last_hidden_state.mean(dim=1)

        # CPU로 이동해서 RAM 안정화
        all_embeddings.append(batch_embeddings.cpu())

        # 메모리 정리 (중요)
        del inputs, outputs, batch_embeddings
        torch.cuda.empty_cache()

    return torch.cat(all_embeddings, dim=0)


# 🔥 문장 수 제한 (안정성 핵심)
sentences = sentences[:2000]

embeddings = embed(sentences, batch_size=8)

In [5]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def select_important(sentences, embeddings, top_n=100):
    centroid = embeddings.mean(dim=0)
    sims = cosine_similarity(embeddings, centroid.unsqueeze(0)).flatten()

    top_idx = np.argsort(sims)[-top_n:]
    return " ".join([sentences[i] for i in sorted(top_idx)])

important_text = select_important(sentences, embeddings)

In [ ]:
from transformers import BartTokenizer, BartForConditionalGeneration

model_name = "facebook/bart-large-cnn"

tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name)

def summarize(text):
    inputs = tokenizer(text, return_tensors="pt", max_length=1024, truncation=True)

    summary_ids = model.generate(
        inputs["input_ids"],
        max_length=1000,
        min_length=500,
        num_beams=4,
        length_penalty=1.0
    )

    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

english_summary = summarize(important_text)
print("=== 영어 요약 ===\n", english_summary)

In [ ]:
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast

model_name = "facebook/mbart-large-50-many-to-many-mmt"

tokenizer = MBart50TokenizerFast.from_pretrained(model_name)
model = MBartForConditionalGeneration.from_pretrained(model_name)

def translate_to_korean(text):
    tokenizer.src_lang = "en_XX"
    encoded = tokenizer(text, return_tensors="pt")

    generated = model.generate(
        **encoded,
        forced_bos_token_id=tokenizer.lang_code_to_id["ko_KR"]
    )

    return tokenizer.decode(generated[0], skip_special_tokens=True)

korean_summary = translate_to_korean(english_summary)

print("\n=== 한국어 요약 ===\n")
print(korean_summary)